# LoRA e QLoRA: execução do exemplo no Colab (GPU T4)

Este notebook roda o código de `code/lora_qlora.py` e gera tudo o que vai para `results/`.
Não há demo ao vivo: rodar uma vez antes da apresentação e baixar os resultados.

**Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`.

Tempo estimado: alguns minutos por configuração (4 configurações + medição de memória).

## 1. Conferir a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Obter o código

Opção A: preencher `REPO_URL` com o endereço do repositório (se estiver no GitHub).
Opção B: deixar `REPO_URL` vazio e enviar os arquivos `lora_qlora.py`, `gerar_figuras.py` e `requirements.txt` da pasta `code/` quando o Colab pedir.

In [ ]:
import os
from pathlib import Path

REPO_URL = ""  # ex.: "https://github.com/<usuario>/trabalho-nlp.git"
PASTA = Path("/content/trabalho-nlp")

if REPO_URL:
    if not PASTA.exists():
        !git clone {REPO_URL} {PASTA}
else:
    from google.colab import files
    (PASTA / "code").mkdir(parents=True, exist_ok=True)
    os.chdir(PASTA / "code")
    enviados = files.upload()  # selecionar os 3 arquivos de code/
    print("enviados:", list(enviados))

os.chdir(PASTA)
!ls code

## 3. Instalar as dependências (versões fixadas)

In [ ]:
!pip install -q -r code/requirements.txt

## 4. Memória do modelo: 16 bits × 4 bits (slide de [QUANTIZAÇÃO])

Grava `results/memoria_modelo.json`.

In [ ]:
!python code/lora_qlora.py --medir-memoria

## 5. Treinos

Cada configuração roda num processo separado, então a memória da GPU é liberada entre uma e outra e o pico medido não se mistura.

| Config | Base | r |
|---|---|---|
| `lora16` | 16 bits (fp16) | 16 |
| `qlora_r16` | NF4 (4 bits) | 16 |
| `qlora_r4` | NF4 | 4 |
| `qlora_r64` | NF4 | 64 |

In [ ]:
!python code/lora_qlora.py --config qlora_r16

In [ ]:
!python code/lora_qlora.py --config lora16

In [ ]:
!python code/lora_qlora.py --config qlora_r4

In [ ]:
!python code/lora_qlora.py --config qlora_r64

## 6. Figuras e tabela agregada

In [ ]:
!python code/gerar_figuras.py
from IPython.display import Image, Markdown, display
display(Image("results/curva_perda.png"))
display(Markdown(open("results/tabela_resultados.md", encoding="utf-8").read()))

## 7. Conferências rápidas (para os slides de adaptadores e parâmetros treináveis)

In [ ]:
import json
for c in ["lora16", "qlora_r16", "qlora_r4", "qlora_r64"]:
    p = json.load(open(f"results/{c}/parametros.txt", encoding="utf-8"))
    print(f"{c:10s} {p['print_trainable_parameters']}")
    print(f"{'':10s} contagens batem: {p['contagens_batem']} | só lora_A/lora_B: {p['so_lora_A_e_lora_B']}")

print()
print(open("results/qlora_r16/modelo_print.txt", encoding="utf-8").read().split("=" * 80)[0])
print(json.dumps(json.load(open("results/qlora_r16/respostas.json", encoding="utf-8")), indent=2, ensure_ascii=False))

## 8. Baixar os resultados

Gera `results.zip` sem os pesos dos adaptadores maiores (eles ficam no Colab; os tamanhos já estão em `metricas.json`).
Descompactar na raiz do repositório local.

In [ ]:
!cd /content/trabalho-nlp && zip -qr results.zip results -x "results/*/checkpoints/*" "results/*/adaptador/*.safetensors"
from google.colab import files
files.download("/content/trabalho-nlp/results.zip")